# PP-OCRv4 Mobile Det 訓練（Colab GPU）

本筆記本用於在 Google Colab 上訓練 PP-OCRv4 mobile **文字偵測 (det)** 模型。

**GitHub Repo:** `https://github.com/install88/trainingOcr`

**流程：**
1. Clone repo + 安裝環境
2. 上傳標注好的資料集（圖片 + label txt）
3. 下載預訓練模型
4. 訓練 det 模型
5. 匯出推論模型 → 轉 ONNX
6. 下載訓練好的模型

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. Clone Repo + 安裝環境

In [ ]:
import os
os.chdir("/content")

# Clone 專案 repo
if not os.path.exists("trainingOcr"):
    !git clone https://github.com/install88/trainingOcr.git

# Clone PaddleOCR 訓練框架
if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

# 安裝 PaddlePaddle GPU 版（Colab 通常是 CUDA 12.x）
# 如果 Colab 用的是 CUDA 11.8，把 cu126 改成 cu118
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 安裝其他依賴
!pip install paddleocr paddle2onnx onnxruntime shapely pyclipper imgaug lmdb

In [ ]:
# 驗證安裝
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")

In [ ]:
import os
os.chdir("/content")

if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

os.chdir("/content/PaddleOCR")
!pwd

In [ ]:
# 方法 A：從 Google Drive 載入（推薦）
from google.colab import drive
drive.mount('/content/drive')

# 假設資料集 zip 放在 Google Drive 的 My Drive/ocr_project/ 下
DRIVE_DATASET_PATH = "/content/drive/MyDrive/ocr_project/det_dataset.zip"

!mkdir -p /content/dataset/det
!unzip -o "{DRIVE_DATASET_PATH}" -d /content/dataset/det/

## 3. 上傳資料集

repo 已 clone 到 `/content/trainingOcr/`，但圖片和標注不在 repo 裡（太大）。

你需要上傳 `det_dataset.zip`（由本機 `split_dataset.py` 產生的 `dataset/det/` 打包）：
```
det_dataset.zip
├── train/           ← 訓練圖片
├── val/             ← 驗證圖片
├── train_label.txt  ← 訓練標注
└── val_label.txt    ← 驗證標注
```

In [ ]:
# 檢查資料集
!echo "=== 資料集結構 ==="
!ls -la /content/dataset/det/
!echo ""
!echo "=== 訓練集圖片數量 ==="
!ls /content/dataset/det/train/ | wc -l
!echo "=== 驗證集圖片數量 ==="
!ls /content/dataset/det/val/ | wc -l
!echo ""
!echo "=== train_label.txt 前 3 行 ==="
!head -3 /content/dataset/det/train_label.txt
!echo ""
!echo "=== val_label.txt 前 3 行 ==="
!head -3 /content/dataset/det/val_label.txt

## 4. 下載預訓練模型

In [ ]:
!mkdir -p /content/pretrained_models

# 下載 PPLCNetV3 backbone 預訓練權重
!wget -nc -P /content/pretrained_models/ \
    https://paddleocr.bj.bcebos.com/pretrained/PPLCNetV3_x0_75_ocr_det.pdparams

print("預訓練模型下載完成！")
!ls -la /content/pretrained_models/

## 5. 建立訓練設定檔

In [ ]:
config_content = """
Global:
  model_name: PP-OCRv4_mobile_det
  debug: false
  use_gpu: true
  epoch_num: 200
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: /content/output/det/
  save_epoch_step: 10
  eval_batch_step:
  - 0
  - 500
  cal_metric_during_train: false
  checkpoints:
  pretrained_model: /content/pretrained_models/PPLCNetV3_x0_75_ocr_det
  save_inference_dir: /content/output/det/inference/
  use_visualdl: true
  infer_img:
  save_res_path: /content/output/det/predicts_db.txt
  d2s_train_image_shape: [3, 640, 640]
  distributed: false

Architecture:
  model_type: det
  algorithm: DB
  Transform: null
  Backbone:
    name: PPLCNetV3
    scale: 0.75
    det: True
  Neck:
    name: RSEFPN
    out_channels: 96
    shortcut: True
  Head:
    name: DBHead
    k: 50
    fix_nan: True

Loss:
  name: DBLoss
  balance_loss: true
  main_loss_type: DiceLoss
  alpha: 5
  beta: 10
  ohem_ratio: 3

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 5.0e-05

PostProcess:
  name: DBPostProcess
  thresh: 0.3
  box_thresh: 0.6
  max_candidates: 1000
  unclip_ratio: 1.5

Metric:
  name: DetMetric
  main_indicator: hmean

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /content/dataset/det/train/
    label_file_list:
      - /content/dataset/det/train_label.txt
    ratio_list: [1.0]
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - DetLabelEncode: null
    - CopyPaste: null
    - IaaAugment:
        augmenter_args:
        - type: Fliplr
          args:
            p: 0.5
        - type: Affine
          args:
            rotate:
            - -10
            - 10
        - type: Resize
          args:
            size:
            - 0.5
            - 3
    - EastRandomCropData:
        size:
        - 640
        - 640
        max_tries: 50
        keep_ratio: true
    - MakeBorderMap:
        shrink_ratio: 0.4
        thresh_min: 0.3
        thresh_max: 0.7
        total_epoch: 200
    - MakeShrinkMap:
        shrink_ratio: 0.4
        min_text_size: 8
        total_epoch: 200
    - NormalizeImage:
        scale: 1./255.
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
        order: hwc
    - ToCHWImage: null
    - KeepKeys:
        keep_keys:
        - image
        - threshold_map
        - threshold_mask
        - shrink_map
        - shrink_mask
  loader:
    shuffle: true
    drop_last: false
    batch_size_per_card: 8
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /content/dataset/det/val/
    label_file_list:
      - /content/dataset/det/val_label.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - DetLabelEncode: null
    - DetResizeForTest:
    - NormalizeImage:
        scale: 1./255.
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
        order: hwc
    - ToCHWImage: null
    - KeepKeys:
        keep_keys:
        - image
        - shape
        - polys
        - ignore_tags
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 1
    num_workers: 2
profiler_options: null
"""

with open("/content/det_finetune.yml", "w") as f:
    f.write(config_content.strip())

print("設定檔已建立：/content/det_finetune.yml")

## 5. 使用 repo 的訓練設定檔

直接用 repo 裡的 config，只需覆寫路徑為 Colab 環境的路徑。

In [ ]:
# 確認 repo config 存在
REPO_DIR = "/content/trainingOcr"
CONFIG_PATH = f"{REPO_DIR}/configs/det/PP-OCRv4_mobile_det_finetune.yml"
!cat {CONFIG_PATH} | head -20
print(f"\n設定檔路徑：{CONFIG_PATH}")

In [ ]:
# [可選] 從 checkpoint 恢復訓練（如果斷線了）
# !python tools/train.py \
#     -c /content/det_finetune.yml \
#     -o Global.checkpoints=/content/output/det/latest

In [ ]:
os.chdir("/content/PaddleOCR")

# 訓練時用 -o 覆寫 config 裡的路徑為 Colab 路徑
!python tools/train.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/pretrained_models/PPLCNetV3_x0_75_ocr_det \
       Global.save_model_dir=/content/output/det/ \
       Global.save_inference_dir=/content/output/det/inference/ \
       Train.dataset.data_dir=/content/dataset/det/train/ \
       Train.dataset.label_file_list=[/content/dataset/det/train_label.txt] \
       Eval.dataset.data_dir=/content/dataset/det/val/ \
       Eval.dataset.label_file_list=[/content/dataset/det/val_label.txt]

In [ ]:
!python tools/eval.py \
    -c /content/det_finetune.yml \
    -o Global.pretrained_model=/content/output/det/best_accuracy

## 8. 匯出推論模型

In [ ]:
!python tools/export_model.py \
    -c /content/det_finetune.yml \
    -o Global.pretrained_model=/content/output/det/best_accuracy \
       Global.save_inference_dir=/content/output/det/inference/

print("推論模型已匯出！")
!ls -la /content/output/det/inference/

## 9. 轉換為 ONNX

In [ ]:
!mkdir -p /content/output/det/onnx

!python -m paddle2onnx.convert \
    --model_dir /content/output/det/inference/ \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file /content/output/det/onnx/ch_PP-OCRv4_det_infer.onnx \
    --opset_version 11 \
    --enable_onnx_checker True

print("ONNX 轉換完成！")
!ls -la /content/output/det/onnx/

In [ ]:
# 驗證 ONNX 模型
import onnxruntime as ort
session = ort.InferenceSession("/content/output/det/onnx/ch_PP-OCRv4_det_infer.onnx")
for inp in session.get_inputs():
    print(f"Input:  {inp.name} shape={inp.shape} type={inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name} shape={out.shape} type={out.type}")
print("\nONNX 模型驗證通過！")

## 10. 下載模型

In [ ]:
# 方法 A：儲存到 Google Drive
!cp -r /content/output/det/ /content/drive/MyDrive/ocr_project/output_det/
print("模型已儲存到 Google Drive！")

In [ ]:
# 方法 B：直接下載 ONNX 檔案
from google.colab import files
files.download("/content/output/det/onnx/ch_PP-OCRv4_det_infer.onnx")

In [ ]:
# 方法 C：打包整個 output 下載
# !cd /content && tar czf det_output.tar.gz output/det/
# files.download("/content/det_output.tar.gz")